In [1]:
import os
import cv2
import torch
import supervision as sv

from ultralytics import YOLO

from tqdm import tqdm

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    device = 0
    print("GPU Available")
    print("GPU:", torch.cuda.get_device_name(0))

else:
    device = "cpu"
    print("Running on CPU")

PyTorch version: 2.6.0+cu124
GPU Available
GPU: NVIDIA GeForce GTX 1650


In [3]:
folders = [
    "../videos",
    "../outputs",
    "../models"
]


for folder in folders:
    os.makedirs(folder, exist_ok=True)


print("Project folders ready")

Project folders ready


In [4]:
model = YOLO("yolo11n.pt")

print("YOLO11 loaded")

YOLO11 loaded


In [5]:
tracker = sv.ByteTrack()

print("ByteTrack initialized")

ByteTrack initialized


C:\Users\Acer\AppData\Local\Temp\ipykernel_26424\3879445296.py:1: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack()


In [6]:
video_path = "../videos/traffic.mp4"


if not os.path.exists(video_path):
    raise FileNotFoundError(
        "Video not found. Put your video inside videos folder."
    )


print("Video found")

Video found


In [7]:
cap = cv2.VideoCapture(video_path)


if not cap.isOpened():
    raise RuntimeError(
        "Cannot open video"
    )


print("Video opened successfully")

Video opened successfully


In [8]:
width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)


fps = cap.get(
    cv2.CAP_PROP_FPS
)


total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)


duration = total_frames / fps


print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Frames:", total_frames)
print("Duration:", round(duration,2), "seconds")

Resolution: 1920 x 1080
FPS: 30.0
Frames: 1053
Duration: 35.1 seconds


In [9]:
output_path = "../outputs/tracked_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError("Failed to create output video.")

print("Output video will be saved to:")
print(os.path.abspath(output_path))

Output video will be saved to:
d:\Computer Vision Projects\5.Traffic Video Analysis\outputs\tracked_output.mp4


In [10]:
box_annotator = sv.BoxAnnotator()

label_annotator = sv.LabelAnnotator()

print("Annotators ready")

Annotators ready


In [11]:
VEHICLE_CLASSES = {
    1,   # bicycle
    2,   # car
    3,   # motorcycle
    5,   # bus
    7    # truck
}

print(VEHICLE_CLASSES)

{1, 2, 3, 5, 7}


In [12]:
for _ in tqdm(range(total_frames), desc="Processing"):

    success, frame = cap.read()

    if not success:
        break

    # YOLO inference
    result = model.predict(
        frame,
        conf=0.35,
        device=device,
        verbose=False
    )[0]

    # Convert to Supervision format
    detections = sv.Detections.from_ultralytics(result)

    # Keep only vehicle classes
    if len(detections) > 0:
        mask = [
            class_id in VEHICLE_CLASSES
            for class_id in detections.class_id
        ]
        detections = detections[mask]

    # Tracking
    detections = tracker.update_with_detections(detections)

    # Labels
    labels = []

    if detections.tracker_id is not None:

        for class_id, track_id in zip(
            detections.class_id,
            detections.tracker_id
        ):
            labels.append(
                f"{model.names[class_id]}  ID:{track_id}"
            )

    # Draw boxes
    annotated = box_annotator.annotate(
        scene=frame.copy(),
        detections=detections
    )

    # Draw labels
    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=labels
    )

    # Save frame
    writer.write(annotated)

Processing: 100%|██████████| 1053/1053 [00:46<00:00, 22.51it/s]


In [13]:
cap.release()
writer.release()

print("Processing completed.")
print("Output saved to:")
print(os.path.abspath(output_path))

Processing completed.
Output saved to:
d:\Computer Vision Projects\5.Traffic Video Analysis\outputs\tracked_output.mp4


In [14]:
import os

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print(
        "File size:",
        round(os.path.getsize(output_path) / (1024 * 1024), 2),
        "MB"
    )

File exists: True
File size: 242.53 MB
